# Notebook 05 — LayerNorm, Residuals, and the Feed-Forward MLP

*The three pieces that turn a single attention layer into a stackable Transformer Block.*

## What we're doing and why

Notebook 04 left us with: `embed → attn → lm_head`. That's a single shallow layer. To get *depth* — the ability to stack 2, 6, 12, 96 blocks — we need three more pieces:

1. **Residual connections** — `x = x + f(x)`. Without these, deep nets don't train (vanishing gradients, optimization landscape becomes a swamp).
2. **LayerNorm** — normalizes activations per-token to keep magnitudes sane across the network. Without it, residuals accumulate scale and blow up.
3. **Feed-forward MLP** — a 2-layer per-position network. Attention mixes across positions; the MLP mixes across channels at each position. Both kinds of mixing are necessary.

Together these define the canonical **Transformer Block**:

```
x = x + attn(ln(x))     # attention sublayer with pre-norm + residual
x = x + mlp(ln(x))      # MLP sublayer with pre-norm + residual
```

Two sublayers, two residuals, two LayerNorms. *That's the entire block.* Stack N of these and you have a transformer.

### What we'll build in this notebook

- A `FeedForward` module — the per-position MLP.
- A `Block` module — the canonical pre-norm transformer block.
- A `TinyTransformer` model that stacks `N_LAYERS` blocks (default 1, hardware-constrained).
- Side-by-side training: nb04's flat model vs this one's blocked model.
- A small experiment: train 1, 2, 3 layers and see what depth buys you (even though we'll deploy 1).


## Cell 1 — Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

text = Path('../data/tinyshakespeare.txt').read_text().lower()
VOCAB_SIZE = 32
top = [c for c, _ in Counter(text).most_common(VOCAB_SIZE - 1)]
itos = ['<unk>'] + top
stoi = {c: i for i, c in enumerate(itos)}
encode = lambda s: [stoi.get(c, 0) for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

BATCH_SIZE = 64; BLOCK_SIZE = 16; EMBED_DIM = 8
LR = 3e-3; N_STEPS = 5000; EVAL_EVERY = 250

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

print('device:', device)


## Cell 2 — Residual connections, explained

### The form

`x = x + sublayer(x)`. The input *bypasses* the sublayer and is added to its output. The sublayer learns a **residual** — the *change* to apply — rather than the full new representation.

### Why this matters for deep nets

Without residuals, a deep network learns `x_N = f_N(f_{N-1}(...f_1(x_0)))`. Gradient flow has to pass through every `f_i`, multiplying Jacobians. If any layer has small singular values, gradients vanish. If any has large ones, they explode. **Training stalls.**

With residuals, `x_N = x_0 + (f_1 + f_2 + ... + f_N)(stuff)`. The identity path `x_0` gives gradients a direct route from output back to input — they don't have to survive every sublayer's Jacobian. **Training works at depth.**

Historically: in 2015, ResNet showed residuals let you train 152-layer CNNs that previously didn't converge at 30 layers. Transformers inherit this directly. You cannot stack transformer blocks without residual connections — every modern LLM relies on them.

### Why it matters even at depth 1

At depth 1 the residual is mostly cosmetic — it gives the optimizer the freedom to learn a tiny correction rather than the entire next state, which tends to make optimization smoother from the very start. The big win arrives when we add the MLP sublayer (a second residual path) and especially when stacking.


## Cell 3 — LayerNorm, explained

### The form

For a token's representation `x` of dim C:

$$\mu = \frac{1}{C} \sum_i x_i, \quad \sigma^2 = \frac{1}{C} \sum_i (x_i - \mu)^2$$

$$\text{LN}(x)_i = \gamma_i \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta_i$$

Each token gets normalized to **zero mean, unit variance along its channel dim**, then scaled by a learnable per-channel gain `γ` and shifted by a learnable per-channel bias `β`.

### Why?

Residual connections **add things up**. After 12 residual additions, the magnitude of `x` can drift large. Large activations → unstable gradients, optimizer struggles, training diverges.

LayerNorm resets activation scale at the input to each sublayer. The sublayer always sees roughly-unit-variance input, regardless of how big the residual stream has grown. The learnable `γ`, `β` then allow the sublayer to recover any preferred scale.

### Pre-norm vs post-norm (matters for stability)

- **Post-norm** (original 2017): `x = ln(x + sublayer(x))`. Normalize *after* the residual add. Sensitive to learning rate; requires warmup.
- **Pre-norm** (everyone today): `x = x + sublayer(ln(x))`. Normalize *inside* the sublayer's input only. More stable, no warmup needed. **We use this.**

The original Transformer paper used post-norm. The lesson learned over the next 5 years was: pre-norm trains much more reliably. Llama, GPT-2/3/4, Mistral, etc. — all pre-norm.

### LayerNorm vs BatchNorm

BatchNorm normalizes across the batch dim. Doesn't work well for sequence models because (a) batch sizes vary, (b) you can't normalize at inference time the same way as training, (c) different positions in the sequence shouldn't be normalized together. LayerNorm sidesteps all this by normalizing per-token. *That's why every sequence model uses LN, not BN.*


## Cell 4 — The feed-forward MLP, explained

### The form

At each position independently:

$$\text{MLP}(x) = W_2 \cdot \text{ReLU}(W_1 x + b_1) + b_2$$

Where `W_1: embed_dim → 4*embed_dim` (expansion), `W_2: 4*embed_dim → embed_dim` (back down). The **4× expansion ratio** is conventional and survived from the original paper because it works.

### Why a non-linear MLP after attention?

Attention is a *linear* operation in `V`: `out = (softmax stuff) @ V`. Stacking pure attention layers without non-linearity collapses to a single (more complex) linear function of the input. The MLP provides:

1. **Non-linearity** (ReLU / GELU). Without this, the whole transformer reduces to attention-flavored linear regression.
2. **Per-position computation**. Attention mixes information *across positions*; the MLP processes each position's mixed representation *independently* through a non-linear function. Both kinds of compute are necessary.

You can think of it as: "attention decides what information each position should have; the MLP decides what to *do* with that information at that position."

### Where most of a transformer's parameters live

In GPT-2 small: ~2/3 of all parameters are in the MLPs (the 4× expansion is expensive). At our embed_dim=8 scale: MLP has `8×32 + 32×8 = 512` params, attention has ~256 — same story.

### Modern variants (FYI)

- **GELU** instead of ReLU (GPT-2 onwards) — smoother, slightly better.
- **SwiGLU** (Llama, PaLM) — adds a gating mechanism; ~1/3 better than ReLU MLP for the same params.

At our scale ReLU is fine. We're not chasing 0.01 nats.


In [ ]:
class Head(nn.Module):
    def __init__(self, embed_dim, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
    def forward(self, x):
        B, T, _ = x.shape
        k = self.key(x); q = self.query(x); v = self.value(x)
        s = q @ k.transpose(-2, -1) / (self.head_size ** 0.5)
        s = s.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        return F.softmax(s, dim=-1) @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        head_size = embed_dim // num_heads
        self.heads = nn.ModuleList([Head(embed_dim, head_size, block_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(embed_dim, embed_dim)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self, embed_dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, mult * embed_dim),
            nn.ReLU(),
            nn.Linear(mult * embed_dim, embed_dim),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    '''Pre-norm transformer block: x = x + attn(ln(x)); x = x + mlp(ln(x)).'''
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, block_size)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = FeedForward(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Cell 5 — The full model

`embed → N × Block → ln_final → lm_head`.

One more LayerNorm at the very end (`ln_final`) — standard practice, normalizes the final residual stream before the unembedding. Cheap, helps stability.


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size, n_layers):
        super().__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[Block(embed_dim, num_heads, block_size) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.lm_head  = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embed(idx) + self.pos_embed(torch.arange(T, device=idx.device))
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

def train_model(model, n_steps=N_STEPS):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    hist = []
    for step in range(n_steps + 1):
        if step % EVAL_EVERY == 0:
            model.eval()
            with torch.no_grad():
                ls = {}
                for split in ('train', 'val'):
                    a = torch.zeros(20)
                    for k in range(20):
                        xb, yb = get_batch(split); _, l = model(xb, yb); a[k] = l.item()
                    ls[split] = a.mean().item()
            hist.append((step, ls['train'], ls['val']))
            model.train()
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return hist


## Cell 6 — Depth experiment: 1, 2, 3 layers

Train three models with same `embed_dim=8, num_heads=1`, varying depth. We'll deploy depth-1 on the 6502, but it's important to see what depth buys you so you understand what's being sacrificed.

Expect:
- **1 layer** ≈ nb04 single-head + small bump from MLP/LN. Maybe ~2.20.
- **2 layers** noticeably better — composition of attention. Maybe ~2.10.
- **3 layers** diminishing returns at this tiny scale. Maybe ~2.05.

At larger scales, depth is one of the biggest levers. At our scale it's modest because embed_dim=8 is the real bottleneck.


In [ ]:
results = {}
for n_layers in (1, 2, 3):
    torch.manual_seed(1337)
    m = TinyTransformer(VOCAB_SIZE, EMBED_DIM, num_heads=1, block_size=BLOCK_SIZE, n_layers=n_layers).to(device)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'\n=== {n_layers}-layer ({n_params} params) ===')
    results[n_layers] = (train_model(m), m, n_params)

plt.figure(figsize=(9, 4))
for n_layers, (hist, _, n_params) in results.items():
    steps, _, val = zip(*hist)
    plt.plot(steps, val, label=f'{n_layers} layer ({n_params} params)')
plt.axhline(2.44, color='red', linestyle='--', alpha=0.5, label='bigram baseline')
plt.xlabel('step'); plt.ylabel('val loss')
plt.title('Depth ablation at embed_dim=8, 1 head')
plt.legend(); plt.grid(alpha=0.3); plt.show()

for n_layers, (hist, _, _) in results.items():
    print(f'{n_layers}-layer final val: {hist[-1][2]:.4f}')


## Cell 7 — Generate from the 1-layer model (the hardware target)

This is the architecture we'll ship to the 6502 (after quantization in nb07). Generate text and see how it compares to nb03's output.


In [ ]:
_, model, _ = results[1]

@torch.no_grad()
def generate(model, prompt='\n', max_new_tokens=400):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print(generate(model))


## Post-mortem

### What you now have

A complete, canonical transformer block. The pattern `x = x + sublayer(ln(x))` is what every modern LLM is built from. You've just implemented it from scratch.

### The three pieces, recap

| Piece | Role | What breaks without it |
|---|---|---|
| Residual | Gradient highway through depth | Deep models don't train; loss diverges |
| LayerNorm | Per-token activation scaling | Activations explode through stacked residuals |
| MLP | Per-position non-linear processing | Stacked attention collapses to linear |

### What's left for the hardware port

- **Notebook 06**: scale up to a "real" model (d=64, 4 heads, multiple layers) and confirm the architecture produces actual English. This is the sanity check — if we can't make a *medium* version work, the *tiny* hardware version won't either.
- **Notebook 07**: shrink back to d=8, then quantize to int8 weights, and pack into the EEPROM binary format. After that we're done with PyTorch and we go to C / 6502 assembly.
